# RNAfish Signal Intensity Quantification

Segments nuclei from DAPI fluorescence images and quantifies mean/total RNAfish signal intensity per cell across three channels (anti-47S, DAPI, IFNB1). Transcriptional foci are detected using a watershed-based algorithm. Results are exported to CSV for downstream statistical analysis.

## 1. Dependencies

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import peak_local_max

## 2. Configuration

In [ ]:
# --- Input ---
directory = "analysis1_npy"  # Input folder; each .npy file is a (3, H, W) array: [anti-47S, DAPI, IFNB1]

# --- Group mapping ---
group_map = {
    '1': 'mock 32H',
    '2': 'GS 32H',
    '5': 'mock 48H',
    '6': 'GS 48H',
}
order = ['1', '2', '5', '6']

# --- Nucleus segmentation parameters ---
global_blur_size = (55, 55)     # Gaussian blur size for DAPI before thresholding
specific_blur_size = (65, 65)   # Blur within each ROI to refine nuclear mask
min_nucleus_area = 1500  # Minimum valid nucleus area (pixels)
min_circularity = 0.65   # Shape filter to exclude elongated or fragmented objects (established range: 0.6–0.7)

# --- Foci detection parameters ---
foci_min_distance = 5    # Minimum pixel distance between detected foci peaks
foci_threshold_rel = 0.2  # Relative intensity threshold for foci detection
otsu_scale_factor = 0.5   # Fraction of Otsu threshold used for binary mask creation

## 3. Image Processing Pipeline

In [ ]:
# ===============================================================
# INITIALIZATION
# ===============================================================

# Map all .npy files to prefixes (e.g., "1_sample1.npy" -> "1_sample1")
prefix_map = {
    fname.rsplit(".", 1)[0]: fname
    for fname in os.listdir(directory)
    if fname.endswith(".npy")
}
complete_sets = list(prefix_map.keys())

# Store per-nucleus and per-group results
results = []

by_group = {
    k: {
        'area': [],
        'mean_dapi': [], 'total_dapi': [],
        'mean_anti47s': [], 'total_anti47s': [],
        'mean_ifnb1': [], 'total_ifnb1': [],
        'n47s': [], 'nifnb1': []
    }
    for k in group_map
}

# ===============================================================
# HELPER FUNCTION — watershed foci counter
# ===============================================================
def count_foci_watershed(channel_roi, mask, min_distance=foci_min_distance, threshold_rel=foci_threshold_rel):
    # Mask the input region
    roi = cv2.bitwise_and(channel_roi, channel_roi, mask=mask)

    # Thresholding with scaled Otsu value
    otsu_val, _ = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adjusted = int(otsu_scale_factor * otsu_val)
    _, thresh = cv2.threshold(roi, adjusted, 255, cv2.THRESH_BINARY)

    # Distance transform for watershed seeds
    dist = cv2.distanceTransform(thresh, cv2.DIST_L2, 3)
    dist = cv2.normalize(dist, None, 0, 1.0, cv2.NORM_MINMAX)

    # Find local maxima (potential foci centers)
    coords = peak_local_max(dist, min_distance=min_distance, threshold_rel=threshold_rel)
    peaks = np.zeros_like(dist, dtype=np.uint8)
    for y, x in coords:
        peaks[y, x] = 255

    # Label and perform watershed
    num_markers, markers = cv2.connectedComponents(peaks)
    markers = markers + 1  # Ensure background != 0
    roi_color = cv2.cvtColor(roi, cv2.COLOR_GRAY2BGR)
    cv2.watershed(roi_color, markers)

    # Extract foci mask and count contours
    foci_mask = np.uint8(markers > 1) * 255
    contours, _ = cv2.findContours(foci_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return len(contours), contours

# ===============================================================
# MAIN LOOP — process all images and nuclei
# ===============================================================
all_areas = []
circularities = []
for prefix in complete_sets:
    group_key = prefix[0]
    if group_key not in group_map:
        continue  # skip unrecognized prefixes

    # --- Load data ---
    path = os.path.join(directory, prefix_map[prefix])
    data = np.load(path)  # shape: (3, H, W)
    anti47s = np.clip(data[0], 0, 255).astype(np.uint8)
    dapi    = np.clip(data[1], 0, 255).astype(np.uint8)
    ifnb1   = np.clip(data[2], 0, 255).astype(np.uint8)

    height, width = dapi.shape

    # --- Global nucleus segmentation on DAPI ---
    blurred = cv2.GaussianBlur(dapi, global_blur_size, 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    areas = [cv2.contourArea(cnt) for cnt in contours]
    all_areas.extend(areas)

    def is_fully_inside(cnt):
        x, y, w, h = cv2.boundingRect(cnt)
        return x > 0 and y > 0 and (x + w) < width and (y + h) < height

    # Filter by area and border proximity
    filtered_contours = [
        cnt for cnt in contours
        if cv2.contourArea(cnt) >= min_nucleus_area and is_fully_inside(cnt)
    ]

    # --- Per-nucleus processing ---
    for j, cnt in enumerate(filtered_contours):
        # Bounding box + padding
        x, y, w, h = cv2.boundingRect(cnt)
        pad_x = int(w * 0.25)
        pad_y = int(h * 0.25)

        x1 = max(x - pad_x, 0)
        y1 = max(y - pad_y, 0)
        x2 = min(x + w + pad_x, width)
        y2 = min(y + h + pad_y, height)

        # Crop ROIs
        roi_dapi    = dapi[y1:y2, x1:x2]
        roi_anti47s = anti47s[y1:y2, x1:x2]
        roi_ifnb1   = ifnb1[y1:y2, x1:x2]

        # Secondary blur refines the nuclear boundary within the cropped ROI
        roi_blurred = cv2.GaussianBlur(roi_dapi, specific_blur_size, 0)
        _, roi_thresh = cv2.threshold(roi_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        roi_contours, _ = cv2.findContours(roi_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not roi_contours:
            continue

        # Use the contour we zoomed in on
        roi_contours = sorted(roi_contours, key=cv2.contourArea, reverse=True)
        largest_cnt = roi_contours[0]

        mask_inside = np.zeros(roi_dapi.shape, dtype=np.uint8)
        cv2.drawContours(mask_inside, [largest_cnt], -1, 255, -1)

        # --- Shape filtering ---
        area = np.count_nonzero(mask_inside)
        if area < min_nucleus_area:
            continue
        perimeter = cv2.arcLength(largest_cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * (area / (perimeter ** 2))
        circularities.append(circularity)
        if circularity < min_circularity:
            continue

        # --- Intensity measurements ---
        mean_dapi    = cv2.mean(roi_dapi,    mask=mask_inside)[0]
        mean_anti47s = cv2.mean(roi_anti47s, mask=mask_inside)[0]
        mean_ifnb1   = cv2.mean(roi_ifnb1,   mask=mask_inside)[0]

        total_dapi    = cv2.sumElems(cv2.bitwise_and(roi_dapi,    roi_dapi,    mask=mask_inside))[0]
        total_anti47s = cv2.sumElems(cv2.bitwise_and(roi_anti47s, roi_anti47s, mask=mask_inside))[0]
        total_ifnb1   = cv2.sumElems(cv2.bitwise_and(roi_ifnb1,   roi_ifnb1,   mask=mask_inside))[0]

        # --- Foci counting ---
        n47s, _   = count_foci_watershed(roi_anti47s, mask_inside)
        nifnb1, _ = count_foci_watershed(roi_ifnb1, mask_inside)

        # --- Record nucleus results ---
        label = f"{prefix}_cell{{j}}"
        results.append((
            label,
            area,
            mean_dapi, total_dapi,
            mean_anti47s, total_anti47s,
            mean_ifnb1, total_ifnb1,
            n47s, nifnb1
        ))

        # Aggregate into per-group dict
        g = by_group[group_key]
        g['area'].append(area)
        g['mean_dapi'].append(mean_dapi)
        g['total_dapi'].append(total_dapi)
        g['mean_anti47s'].append(mean_anti47s)
        g['total_anti47s'].append(total_anti47s)
        g['mean_ifnb1'].append(mean_ifnb1)
        g['total_ifnb1'].append(total_ifnb1)
        g['n47s'].append(n47s)
        g['nifnb1'].append(nifnb1)

## 4. Export Results

In [ ]:
# ===============================================================
# SAVE DATA
# ===============================================================

columns = [
    "label", "area",
    "mean_dapi", "total_dapi",
    "mean_anti47s", "total_anti47s",
    "mean_ifnb1", "total_ifnb1",
    "n47s", "nifnb1"
]
df = pd.DataFrame(results, columns=columns)
df["group_key"] = df["label"].str[0]
df["condition"] = df["group_key"].map(group_map)
df.to_csv("image1_results.csv", index=False)